# EnergeticPricing — prévision du prix day-ahead français

Pipeline complet : chargement des données (météo par département, prévisions ENTSO-E, gaz TTF, capacités installées), entraînement avec validation temporelle, puis prévision d'une journée.

Prérequis : une clé ENTSO-E dans `.env` (cf. `.env.example`).

In [ ]:
from src.market.market import EnergyMarket

## Paramètres

In [ ]:
START = "2023-01-01"
END = "2026-05-01"
FREQ = "1h"                # "1h" ou "15min"
METEO_MODE = "department"  # "department" (météo par dép.) ou "national" (moyenne pondérée pop.)
MODEL = "LightGBM"         # "Simple" (Ridge), "RandomForest", "GradientBoosting", "LightGBM", "TFT"

## Chargement des données
Les appels API sont mis en cache dans `data/cache/` (parquet).

In [ ]:
market = EnergyMarket(start=START, end=END, freq=FREQ, meteo_mode=METEO_MODE)
market.initialize()

## Entraînement
Validation rolling-origin (TimeSeriesSplit, 5 plis). `use_neighbour_prices=True` ajoute les prix DA voisins : meilleur score, mais ils ne sont pas connus la veille (couplage EUPHEMIA) -> à réserver à l'analyse a posteriori.

In [ ]:
market.learn(MODEL, use_neighbour_prices=False)

## Prévision d'une journée
Jusqu'à J+3 ; le prix réel est affiché s'il est publié.

In [ ]:
market.predict_from_meteo("2026-04-28")